# 7. Final Evaluation

## Purpose
Evaluate the final tuned model on the held out test set for the first
and only time.

## Inputs
- `data/processed/` — held out test features and labels
- `data/raw/` — raw data reloaded for CatBoost
- `config.yaml` — model selection and tuning settings
- `artifacts/baseline_trainer.pkl` — baseline CV scores for comparison
- `artifacts/tuning_results.pkl` — tuned CV scores for comparison
- `artifacts/tuned_models.pkl` — final tuned model
- `artifacts/threshold.pkl` — optimal classification threshold

## Outputs
- `models/final/` — definitive model saved for deployment
- `reports/final_evaluation.json` — machine readable metrics report
- `reports/final_evaluation.html` — human readable metrics report
- `reports/figures/workflow_comparison.png` — baseline vs tuned vs
  final test score chart
- `reports/figures/roc_curve.png` — ROC curve (classification)
- `reports/figures/precision_recall_curve.png` — PR curv

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.visualization.evaluation import (
    plot_roc_curve,
    plot_precision_recall_curve,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_predicted_vs_actual,
    plot_residuals,
    plot_residual_distribution,
)

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    ARTIFACTS_DIR,
    FINAL_MODELS_DIR,
    REPORTS_DIR,
    FIGURES_DIR
)
from src.data.load_data import load_raw_data
from src.models.training.metrics import (
    compute_classification_metrics,
    compute_regression_metrics,
)
from src.models.training.save_load import save_model
from src.utils.reporting import save_json_report, save_html_report

## 1. Load config

In [ ]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
TUNING_CONFIG = config["tuning"]
THRESHOLD_CONFIG = config["threshold_tuning"]

print(f"Target:         {TARGET}")
print(f"Model:          {TUNING_CONFIG['models_to_tune'][0]}")

## 2. Load artifacts
All artifacts generated throughout the workflow are loaded here for
use in the final evaluation and reporting steps.

In [ ]:
baseline_trainer = joblib.load(ARTIFACTS_DIR / "baseline_trainer.pkl")
tuning_results = joblib.load(ARTIFACTS_DIR / "tuning_results.pkl")
tuned_models = joblib.load(ARTIFACTS_DIR / "tuned_models.pkl")
threshold_artifact = joblib.load(ARTIFACTS_DIR / "threshold.pkl")

task = baseline_trainer.task_
model_name = TUNING_CONFIG["models_to_tune"][0]
model = tuned_models[model_name]

print(f"Task:           {task}")
print(f"Optimal threshold: {threshold_artifact['optimal_threshold']:.4f} "
      f"(metric: {threshold_artifact['metric']})")

## 3. Load test data
The test set that has been held out since notebook 2 is loaded here
for the first and only time.

In [ ]:
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

# Load raw test data for CatBoost
if "catboost" in model_name:
    from sklearn.model_selection import train_test_split
    from src.models.training.trainer import ModelTrainer

    df = load_raw_data()
    X_raw = df.drop(columns=[TARGET] + DROP_COLS)
    y_all = df[TARGET]

    stratify_col = y_all if SPLIT_CONFIG["stratify"] else None
    _, X_test_raw, _, _ = train_test_split(
        X_raw, y_all,
        test_size=SPLIT_CONFIG["test_size"],
        random_state=SPLIT_CONFIG["random_state"],
        stratify=stratify_col
    )
    trainer_helper = ModelTrainer()
    X_test_input = trainer_helper._prepare_catboost_data(X_test_raw)
else:
    X_test_input = X_test

print(f"Test set shape: {X_test.shape}")

## 4. Get test set predictions
Predictions are computed using the tuned model. For classification,
both default (0.5) and optimal threshold predictions are computed
so they can be compared directly.

In [ ]:
if task == "classification":
    y_proba = model.predict_proba(X_test_input)
    y_proba_pos = y_proba if y_proba.ndim == 1 else y_proba[:, 1]

    # Default threshold predictions
    y_pred_default = (y_proba_pos >= 0.5).astype(int)

    # Optimal threshold predictions
    optimal_threshold = threshold_artifact["optimal_threshold"]
    y_pred_optimal = (y_proba_pos >= optimal_threshold).astype(int)

    print(f"Predictions computed using optimal threshold: {optimal_threshold:.4f}")

else:
    y_pred = model.predict(X_test_input)
    print("Regression predictions computed.")

## 5. Compute final test metrics
All metrics are computed on the held out test set. For classification,
metrics are computed at both the default and optimal threshold so the
impact of threshold tuning is clear.

In [ ]:
if task == "classification":
    metrics_default = compute_classification_metrics(
        y_test.values, y_proba_pos, y_pred_default
    )
    metrics_optimal = compute_classification_metrics(
        y_test.values, y_proba_pos, y_pred_optimal
    )

    print("\nTest metrics — default threshold (0.5):")
    for k, v in metrics_default.items():
        if v is not None:
            print(f"  {k}: {float(v):.4f}")

    print(f"\nTest metrics — optimal threshold ({optimal_threshold:.4f}):")
    for k, v in metrics_optimal.items():
        if v is not None:
            print(f"  {k}: {float(v):.4f}")

    final_metrics = metrics_optimal

else:
    final_metrics = compute_regression_metrics(y_test.values, y_pred)
    print("\nTest metrics:")
    for k, v in final_metrics.items():
        if v is not None:
            print(f"  {k}: {float(v):.4f}")

## 6. Workflow comparison
Collects CV scores from baseline and tuning notebooks alongside the
final test score.

In [ ]:
baseline_row = baseline_trainer.leaderboard_[
    baseline_trainer.leaderboard_["model_name"] == model_name
]
baseline_cv_score = float(baseline_row["primary_metric_value"].values[0]) \
    if len(baseline_row) > 0 else None

tuning_result = tuning_results.get(model_name)
if tuning_result is not None:
    method = TUNING_CONFIG["method"]
    tuned_cv_score = float(tuning_result.best_value) \
        if method == "optuna" else float(tuning_result.best_score_)
else:
    tuned_cv_score = None

primary_metric = baseline_trainer.leaderboard_.iloc[0]["primary_metric_name"]

if task == "classification":
    final_test_score = final_metrics.get("roc_auc")
else:
    final_test_score = final_metrics.get("rmse")

comparison = {
    "Baseline CV": baseline_cv_score,
    "Tuned CV": tuned_cv_score,
    "Final test": final_test_score,
}

print(f"\nWorkflow comparison ({primary_metric}):")
for stage, score in comparison.items():
    if score is not None:
        print(f"  {stage}: {score:.4f}")

## 7. Workflow comparison chart
Visual summary of performance at each stage of the workflow.
Baseline CV → Tuned CV → Final test.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

stages = [k for k, v in comparison.items() if v is not None]
scores = [v for v in comparison.values() if v is not None]
colors = ["#5DCAA5", "#378ADD", "#D85A30"][:len(stages)]

bars = ax.bar(stages, scores, color=colors, width=0.5, edgecolor="none")

# Add value labels on bars
for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{score:.4f}",
        ha="center", va="bottom",
        fontsize=11, fontweight="500"
    )

ax.set_ylabel(primary_metric.upper(), fontsize=12)
ax.set_title(f"Workflow comparison — {model_name}", fontsize=13)
ax.set_ylim(min(scores) * 0.97, max(scores) * 1.03)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / "workflow_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Diagnostic plots
Visualisations to diagnose model behaviour on the test set:
- Classification: ROC curve, precision-recall curve, confusion matrices
  at both default and optimal threshold, feature importance
- Regression: predicted vs actual, residuals, residual distribution

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if task == "classification":
    # ROC curve
    plot_roc_curve(
        y_test.values,
        y_proba_pos,
        save_path=FIGURES_DIR / "roc_curve.png"
    )

    # Precision-Recall curve
    plot_precision_recall_curve(
        y_test.values,
        y_proba_pos,
        save_path=FIGURES_DIR / "precision_recall_curve.png"
    )

    # Confusion matrices — default vs optimal threshold
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    cm_default = confusion_matrix(y_test.values, y_pred_default)
    cm_optimal = confusion_matrix(y_test.values, y_pred_optimal)

    ConfusionMatrixDisplay(cm_default).plot(ax=axes[0], colorbar=False)
    axes[0].set_title("Default threshold (0.5)")

    ConfusionMatrixDisplay(cm_optimal).plot(ax=axes[1], colorbar=False)
    axes[1].set_title(f"Optimal threshold ({optimal_threshold:.2f})")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Feature importance fallback
    plot_feature_importance(
        model,
        list(X_test_input.columns),
        save_path=FIGURES_DIR / "feature_importance.png"
    )

else:
    # Regression plots
    plot_predicted_vs_actual(
        y_test.values, y_pred,
        save_path=FIGURES_DIR / "predicted_vs_actual.png"
    )
    plot_residuals(
        y_test.values, y_pred,
        save_path=FIGURES_DIR / "residuals.png"
    )
    plot_residual_distribution(
        y_test.values, y_pred,
        save_path=FIGURES_DIR / "residual_distribution.png"
    )

## 9. SHAP analysis on test set
SHAP values computed on the held out test set confirm which features
are driving predictions on genuinely unseen data.

In [ ]:
import shap

print("Computing SHAP values on test set")

try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_input)

    if isinstance(shap_values, list):
        shap_vals_plot = shap_values[1]
    else:
        shap_vals_plot = shap_values

    # Summary plot
    plt.figure()
    shap.summary_plot(
        shap_vals_plot,
        X_test_input,
        show=False,
        max_display=15
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_summary_test.png", dpi=150, bbox_inches="tight")
    plt.show()

except Exception as e:
    print(f"SHAP analysis skipped: {e}")

## 10. Save final model
The tuned model is saved to `models/final/` as the definitive artifact
for deployment. This is the model that should be used for any downstream
prediction tasks.

In [ ]:
FINAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)
save_model(model, FINAL_MODELS_DIR / f"{model_name}_final.pkl")
print(f"Final model saved to {FINAL_MODELS_DIR / f'{model_name}_final.pkl'}")

## 11. Save JSON report
Machine readable metrics report saved to `reports/`.

In [ ]:
json_metrics = {
    "model": model_name,
    "task": task,
    "optimal_threshold": float(optimal_threshold) if task == "classification" else None,
    "threshold_metric": THRESHOLD_CONFIG["metric"] if task == "classification" else None,
    "test_metrics": {k: float(v) for k, v in final_metrics.items() if v is not None},
    "workflow_comparison": {k: float(v) for k, v in comparison.items() if v is not None},
}

save_json_report(
    json_metrics,
    REPORTS_DIR / "final_evaluation.json"
)

## 12. Save HTML report
Human readable report saved to `reports/`. Can be shared directly
with business stakeholders without requiring any technical tools to open.

In [ ]:
save_html_report(
    metrics=final_metrics,
    comparison=comparison,
    model_name=model_name,
    task=task,
    threshold=optimal_threshold if task == "classification" else None,
    path=REPORTS_DIR / "final_evaluation.html"
)

print("\nFinal evaluation complete.")
print(f"Reports saved to {REPORTS_DIR}")